In [29]:
import requests
import pandas as pd

url = "http://localhost:8000/upload_config_from_csv/1/"

input_csv = "/Users/ndainoran/Desktop/PORTAL/deidentification/NOTEBOOK/Noran/noran_final.csv"

files = {"file": ("all_tables.csv", open(input_csv, "rb"), "text/csv")}

# response = requests.post(url, files=files)

# print(response.status_code, response.json())

In [31]:
df = pd.read_csv(input_csv)
df.shape

(6699, 7)

In [32]:
tables = list(df['TABLE_NAME'].unique())[:-1]
len(tables)

433

In [44]:
# list(TableDetailsModel.objects.filter(table_name__in=tables, table_status=1).values_list('table_name', flat=True))

In [33]:
import os
import django
import sys
# Set up Django environment
sys.path.append('/Users/ndainoran/Desktop/PORTAL/deidentification/deIdentification/')
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "deIdentification.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()


from nd_api.models import DbDetailsModel, TableDetailsModel, IgnoreRowsDeIdentificaiton
from worker.models import Task, Chain
from django.contrib.auth.models import User
from keycloakauth.rolemodel import RoleModel, get_default_permissions
from nd_scripts.create_account import create_account
from nd_api.views.db_views import create_stats_generation_tasks
from core.process.main import start_de_identification_for_table
from nd_api.views.de_identification_task import create_deidentification_task

def clean_db():
    RoleModel.objects.all().delete()
    User.objects.all().delete()
    DbDetailsModel.objects.all().delete()
    Chain.objects.all().delete()

In [34]:
def fix_reference_value(table_id):
    table = TableDetailsModel.objects.get(id=table_id)
    
    pid_col = [
        col['column_name'] for col in table.table_details_for_ui['columns_details'] if col['is_phi'] and col["de_identification_rule"].startswith('PATIENT_ID')
    ]
    enc_col = [
        col['column_name'] for col in table.table_details_for_ui['columns_details'] if col["de_identification_rule"] == "ENCOUNTER_ID"
    ]
    print(pid_col)
    print(enc_col)
    patient_id = pid_col[0] if len(pid_col)>0 else None
    enc_id = enc_col[0] if len(enc_col)>0 else None
    table.table_details_for_ui['reference_enc_id_column'] = enc_id
    table.table_details_for_ui['reference_patient_id_column'] = patient_id
    table.save()

In [40]:
TableDetailsModel.objects.filter(table_status=0).count()

902

In [42]:
TableDetailsModel.objects.filter(table_status=2).count()

425

In [4]:
TableDetailsModel.objects.filter(table_status=2).count()
TableDetailsModel.objects.get(id=707).table_name

'DIRECTIV'

In [38]:
# 45861
# gt 566
from django.conf import settings
settings.BATCH_SIZE_DURING_DE_IDENTIFICATION

1000

In [49]:
Task.objects.filter(status=0).count()

222011

In [31]:
print(Task.objects.filter(status=-1).count())
print(Task.objects.filter(failure_count__gt=0).count())

1613
3404


In [9]:
Task.objects.filter(failure_count__gt=0).exclude(status=2).count()

3405

In [82]:
Task.objects.filter(failure_count__gt=0).exclude(status=2)[3250].remarks

{}

In [83]:
Task.objects.filter(failure_count__gt=0).exclude(status=2)[3250].arguments['table_id']

790

In [21]:
Task.objects.filter(arguments__table_id=1208, failure_count__gt=0).exclude(status=2).count()

925

In [15]:
TableDetailsModel.objects.get(id=1353)
t = TableDetailsModel.objects.get(table_name = "PatientVisitAttachment")
# t.table_details_for_ui


In [22]:
Task.objects.filter(status=-1).count()

1613

In [45]:
from django.db.models import Count

grouped_tasks = Task.all_objects.filter(status=0).values('arguments__table_id').annotate(count=Count('id'))
#print(list(counts))

table_ids = [entry['arguments__table_id'] for entry in grouped_tasks if entry['arguments__table_id'] is not None]

table_info = TableDetailsModel.objects.filter(id__in=table_ids).values('id', 'table_name')
table_map = {entry['id']: entry['table_name'] for entry in table_info}

#print(table_info)
#print(table_map)

result = []
for entry in grouped_tasks:
    table_id = entry['arguments__table_id']
    count = entry['count']
    table_name = table_map.get(table_id, 'Unknown')
    result.append({'table_id': table_id, 'table_name': table_name, 'count': count})


total_tasks=0
total_tables = []
for r in result:
    total_tasks += r.get('count', 0)
    total_tables.append(r.get('table_name'))
    print("Tasks Left: ", r)
print("Total Tasks:", total_tasks)
print("Total Tables:", len(total_tables))

Tasks Left:  {'table_id': 1208, 'table_name': 'RcopiaSynchronization', 'count': 926}
Tasks Left:  {'table_id': 898, 'table_name': 'Guarantor', 'count': 61}
Tasks Left:  {'table_id': 1094, 'table_name': 'OBS', 'count': 2}
Tasks Left:  {'table_id': 1110, 'table_name': 'PATCHES', 'count': 2}
Tasks Left:  {'table_id': 1158, 'table_name': 'PRESCRIB', 'count': 287}
Tasks Left:  {'table_id': 1034, 'table_name': 'MEDICATE_AUDIT', 'count': 158}
Tasks Left:  {'table_id': 1050, 'table_name': 'MIKPatientVisit', 'count': 2}
Tasks Left:  {'table_id': 789, 'table_name': 'EDIReport', 'count': 5}
Tasks Left:  {'table_id': 1130, 'table_name': 'PatientRelationship', 'count': 170}
Tasks Left:  {'table_id': 1123, 'table_name': 'PatientProfile_BAK', 'count': 36}
Tasks Left:  {'table_id': 1061, 'table_name': 'MM_CEM_EventQueue', 'count': 3}
Tasks Left:  {'table_id': 64, 'table_name': 'ApptSlot', 'count': 3}
Tasks Left:  {'table_id': 1203, 'table_name': 'RcopiaMessage', 'count': 149}
Tasks Left:  {'table_id':

In [53]:
# Chain.objects.all().delete()

"""
EDIReport -> duplicate rows
"""
# tables2 = [
#     "PatientVisitAmbulance",
#     "PatientVisitFilingConditionCodes",
#     "PatientVisitResource",
#     "EDIClaim",
#     "PatientVisitFiling",
#     "PatientVisitInsurance",
#     "PatientVisitProcs",
#     "PatientVisitProcsAgg",
#     "PatientVisitDiags",
#     "VisitTransactions",
#     "EDIReport"
# ]

tables3 = [
    'MM_CEM_EventQueue',
    'PatientProfile_BAK',
    'PRESCRIB',
    'PatientRelationship',
    'PatientProfileAttachment',
    'PatientProfileReport',
    'FamilyHealthHistory',
    'Guarantor',
    'InteractionOverride',
    'MEDICATE_AUDIT',
    'MIKPatientVisit',
    'OBS',
    'ApptSlot',
    'EDIReportFile',
    'FLAG',
    'FlagBackup',
    'PATCHES',
    'RcopiaSynchronization',
    'RcopiaMessage',
    'DOCCONTB'
]

In [54]:
totalrun = 0
for table in tables3:
# for table in tables[::-1]:
# for table in ['PatientProfile']:
    tableobj = TableDetailsModel.objects.get(table_name=table)
    Task.objects.filter(arguments__table_id=tableobj.id).delete()
    # if tableobj.table_status in [2]:
    #     continue
    fix_reference_value(tableobj.id)
    tableobj.refresh_from_db()
    tasks, chain = create_deidentification_task(tableobj, delete_table=True)
    totalrun += 1
    print(tableobj.rows_count)

2025-06-05 06:33:08,729 - deIdentification.nd_logger - INFO - inside marking as in progress if required
2025-06-05 06:33:08,740 - deIdentification.nd_logger - WARNING - Table MM_CEM_EventQueue does not exist and cannot be dropped.
2025-06-05 06:33:08,741 - deIdentification.nd_logger - INFO - Dropping ignore rows for MM_CEM_EventQueue, noren_prod_local


['PID']
[]


2025-06-05 06:33:31,313 - deIdentification.nd_logger - INFO - inside marking as in progress if required
2025-06-05 06:33:31,325 - deIdentification.nd_logger - WARNING - Table PatientProfile_BAK does not exist and cannot be dropped.
2025-06-05 06:33:31,327 - deIdentification.nd_logger - INFO - Dropping ignore rows for PatientProfile_BAK, noren_prod_local


1780
['PatientProfileId', 'PatientId', 'PId']
[]


2025-06-05 06:33:50,775 - deIdentification.nd_logger - INFO - inside marking as in progress if required
2025-06-05 06:33:50,781 - deIdentification.nd_logger - WARNING - Table PRESCRIB does not exist and cannot be dropped.
2025-06-05 06:33:50,781 - deIdentification.nd_logger - INFO - Dropping ignore rows for PRESCRIB, noren_prod_local


...

924350
['PID']
[]


2025-06-05 06:39:50,850 - deIdentification.nd_logger - INFO - inside marking as in progress if required


147623
[]
[]


2025-06-05 06:39:51,128 - deIdentification.nd_logger - INFO - Table DOCCONTB dropped from the database.
2025-06-05 06:39:51,130 - deIdentification.nd_logger - INFO - Dropping ignore rows for DOCCONTB, noren_prod_local


265200124


In [10]:
IgnoreRowsDeIdentificaiton.objects.filter(db_name="noren_prod_local", table_name="PatientVisitPaperClaim")[100].row

'{"PatientVisitPaperClaimId": 314, "PatientVisitId": 73045, "FilingMethodMId": 140, "Charges": {"py/reduce": [{"py/type": "decimal.Decimal"}, {"py/tuple": ["718.0000"]}]}, "Procedures": "99205", "Name": "Premier Access Inc", "ClaimStatus": null, "Created": {"py/object": "datetime.datetime", "__reduce__": [{"py/type": "datetime.datetime"}, ["B+YDHwwMOQHwGA=="]]}, "CreatedBy": "bruce", "LastModified": {"py/object": "datetime.datetime", "__reduce__": [{"py/type": "datetime.datetime"}, ["B+YDHwwMOQHwGA=="]]}, "LastModifiedBy": "bruce", "Diagnoses": "R58 H53.461", "nd_auto_increment_id": 632}'

In [84]:
tb  = TableDetailsModel.objects.get(id=790)
[(c['is_phi'], c['de_identification_rule']) for c in tb.table_details_for_ui['columns_details']]
tb.table_name

'EDIReportFile'

In [33]:
TableDetailsModel.objects.get(table_name="PatientVisitAttachment")

<TableDetailsModel: PatientVisitAttachment - noren_prod_local - 1135>

In [10]:
for task in Task.objects.filter(status=0):
    task.status = 0
    task.failure_count = 0
    task.save()

In [37]:
Task.objects.filter(arguments__table_id=1135).first()

Task(id=606657, type=core.process.main.start_de_identification_for_table, arguments=({'hooks': {'failure'...), status=0, num_dependencies=0, num_dependencies_pending=0, failure_count=0)